# Part 9 — Knowledge-based suitability (7 segments)

FAO land evaluation + AHP: fuzzy-standardize each factor of the **raw** `feature_stack_250m` to [0,1], combine per segment with a **weighted geometric mean** (limiting-factor behavior), mask with the Part-7b land-cover masks, and classify into FAO **S1/S2/S3/N**. Rules live in `config/segments.yaml`; the engine in `src/membership.py`.

**Output:** `suit_present` (7 `suit_*` + 7 `class_*` bands). **DoD:** 7 maps render; built-up/water masked; soy high on flat/fertile land, conservation high on steep/native.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import ee, geemap
import utils, features
project = utils.init()
print('EE initialized; project =', project)
import membership, external

In [ ]:
aoi = utils.load_aoi(project)
print('AOI area (km^2):', round(aoi.area(1000).divide(1e6).getInfo(), 1))

### Load the raw 250 m stack + land-cover masks, the segment rules, and the side-image factors
`extras` routes the recalibrated side-image factors into the engine (`membership._source_image`): `sit_` → `feat_siting` (solar clearness / pisci seasonal-water distance), `rl_` → the realized-use image (conservation's demoted `rl_native_frac`), `cv_` → `feat_conservation` (conservation-value: WDPA distance / ruggedness / biomass carbon). These stay **out** of the 34-band stack (guardrail).

In [ ]:
stack = ee.Image(utils.asset_id(project, 'feature_stack_250m'))
lc = ee.Image(utils.asset_id(project, 'feat_landcover'))
seg_cfg = utils.cfg('segments')
segs = list(seg_cfg['segments'])
# side-image factors (cached feat_siting / feat_conservation once Parts 7c/7d exported)
siting = ee.Image(utils.asset_id(project, 'feat_siting'))
consv = ee.Image(utils.asset_id(project, 'feat_conservation'))
realized = external.realized_features(aoi)
extras = {'sit_': siting, 'rl_': realized, 'cv_': consv}
print('segments:', segs)

### AHP weight sanity — each segment's factor weights sum to 1
(`membership.consistency_ratio(matrix)` is available to back these with a Saaty pairwise matrix when one is elicited.)

In [ ]:
import pandas as pd
rows = [(s, len(d['factors']),
         round(sum(f['weight'] for f in d['factors'].values()), 3), d['mask'])
        for s, d in seg_cfg['segments'].items()]
pd.DataFrame(rows, columns=['segment', 'n_factors', 'weight_sum', 'mask'])

### Build the 7-segment suitability + FAO class image

In [ ]:
suit = membership.suit_present(stack, lc, seg_cfg, extras)
bands = suit.bandNames().getInfo()
print(f'{len(bands)} bands:'); print(bands)

### Comparative best-use + siting-screen variance test (2026-07-14)
`comparative_present` z-normalizes each segment's suitability over the AOI so the unequal-permissiveness segments compare on common footing (`best_use` argmax + `comp_*` bands). `segment_spatial_std` is the σ≲0.15 variance test — segments below it (expected: solar, pisciculture) are presented as feasibility screens, not graded surfaces.

In [ ]:
comp = membership.comparative_present(suit, aoi, segs, scale=1000)
sd = membership.segment_spatial_std(suit, aoi, segs, scale=1000).getInfo()
import pandas as pd
sdf = pd.Series({s: sd.get(f'suit_{s}') for s in segs}).round(3)
print('spatial std per segment (screen if <= 0.15):'); print(sdf)

### DoD sanity — suitability bands must lie in [0,1]

In [ ]:
utils.range_report(suit.select([f'suit_{s}' for s in segs]), aoi)

### Quick look — soybean (flat/fertile) vs conservation (steep/native)

In [ ]:
Map = geemap.Map(); Map.centerObject(aoi, 7)
vis = {'min': 0, 'max': 1, 'palette': ['red', 'yellow', 'green']}
Map.addLayer(suit.select('suit_soybean'), vis, 'soybean')
Map.addLayer(suit.select('suit_conservation'), vis, 'conservation', False)
Map.addLayer(suit.select('suit_solar'), vis, 'solar', False)
Map.addLayer(aoi, {}, 'AOI', False)
Map

### Export `suit_present` (7 suitability + 7 FAO-class bands) + `suit_present_comp`

In [ ]:
utils.ensure_folder(project)
task = utils.export_image(suit, project, 'suit_present', aoi)
# comparative best-use (comp_* + best_use argmax) for the atlas / zone profiling
t_comp = utils.export_image(comp, project, 'suit_present_comp', aoi)
print(utils.task_summary([task, t_comp]))

### (Optional) Weight-sensitivity robustness — ±20% AHP sweep
Heavier compute (re-evaluates each segment under perturbed weights). Submit only if quota allows; smaller band = more robust. Exports a separate asset so the main `suit_present` stays lean.

In [ ]:
sens = membership.sensitivity_present(stack, lc, seg_cfg, extras)
t2 = utils.export_image(sens, project, 'suit_present_sens', aoi)
print('export', t2.status()['description'], '->', t2.status()['state'])